# Exploratory Analysis of Factors Influencing Cognitive Decline
## Classification of Diverse Health Domains Across Regional and Demographic Populations

**Authors:** Ummid Salma Mulla, Pratibha M Patil, P. G. Sunitha Hiremath, Neha Tarannum Pendari  
**Institution:** KLE Technological University, Hubballi, India  
**Dataset:** CDC BRFSS – Alzheimer's Disease and Healthy Aging Data (2015–2022)  
**Data Source:** https://data.cdc.gov/Healthy-Aging/Alzheimer-s-Disease-and-Healthy-Aging-Data/hfr9-rurv/data_preview


## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)
from sklearn.feature_selection import SelectKBest, chi2
from imblearn.over_sampling import SMOTE

# ── global style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})
sns.set_palette('muted')
print("✅ All libraries imported successfully.")


## 2. Load Dataset

In [ ]:
# ── Load the CSV downloaded from CDC ─────────────────────────────────────────
# Place the CSV file in the same directory and update the filename if needed.
CSV_FILE = 'alzheimer_disease_and_healthy_aging_data.csv'   # <-- update if needed

try:
    df = pd.read_csv(CSV_FILE, low_memory=False)
    print(f"Dataset loaded  →  {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
except FileNotFoundError:
    print("⚠️  CSV not found – generating a synthetic demo dataset so all code runs.")
    # ── synthetic data matching paper's description ───────────────────────────
    np.random.seed(42)
    n = 284_142
    years   = np.random.choice(range(2015, 2023), n)
    locs    = [f'Location_{i}' for i in range(1, 60)]
    regions = ['Northeast','Midwest','South','West',
               'Pacific','Mountain','New England','North Central']
    topics  = ['Cognitive Decline','Caregiving','Mental Health','Overall Health']
    age_grp = ['65-69','70-74','75-79','80+']
    gender  = ['Male','Female']
    race    = ['White','Black','Hispanic','Asian','Other']

    df = pd.DataFrame({
        'YearStart'         : years,
        'YearEnd'           : years,
        'LocationAbbr'      : np.random.choice(locs, n),
        'LocationDesc'      : np.random.choice(regions, n),
        'Topic'             : np.random.choice(topics, n,
                                p=[0.12, 0.17, 0.18, 0.53]),
        'Question'          : np.random.choice(
                                ['Talked to doctor','Functional limitations',
                                 'Memory worsening','Caregiver hours',
                                 'Depression screening','Exercise frequency',
                                 'Social isolation','Health status','Sleep quality',
                                 'Medication adherence'], n),
        'Data_Value'        : np.random.uniform(0, 100, n),
        'StratificationCategory1': np.random.choice(['Sex','Race/Ethnicity'], n,
                                                     p=[0.55, 0.45]),
        'Stratification1'   : np.random.choice(gender + race, n),
        'AgeGroup'          : np.random.choice(age_grp, n),
        'Gender'            : np.random.choice(gender, n),
        'Race'              : np.random.choice(race, n),
        'Income'            : np.random.choice(
                                ['<25k','25-50k','50-75k','>75k'], n),
        'Comorbidities'     : np.random.randint(0, 6, n),
        'CaregivingHours'   : np.random.uniform(0, 60, n),
        'PhysicalActivity'  : np.random.choice(['Yes','No'], n),
        'Depression'        : np.random.choice(['Yes','No'], n),
        'Hypertension'      : np.random.choice(['Yes','No'], n),
        'Diabetes'          : np.random.choice(['Yes','No'], n),
    })
    # introduce some missing values
    for col in ['Data_Value','CaregivingHours','Income']:
        mask = np.random.rand(n) < 0.05
        df.loc[mask, col] = np.nan

print(df.head())
print("\nColumn dtypes:")
print(df.dtypes)


## 3. Data Preprocessing

In [ ]:
print("=== Missing Values (before) ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

# ── Impute numerical columns with median ──────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# ── Impute categorical columns with mode ─────────────────────────────────────
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# ── Drop duplicate rows ───────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f"\nDuplicates removed: {before - len(df)}")

print("\n=== Missing Values (after) ===")
print(df.isnull().sum().sum(), "total missing values")
print(f"\nFinal dataset shape: {df.shape}")


In [ ]:
# ── Encode categorical columns for ML ────────────────────────────────────────
le = LabelEncoder()
encode_cols = ['Gender','Race','Income','PhysicalActivity','Depression',
               'Hypertension','Diabetes','StratificationCategory1',
               'Stratification1','AgeGroup','Question','LocationDesc']

for col in encode_cols:
    if col in df.columns:
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))

# ── Target label: Topic (4 health domains) ───────────────────────────────────
df['Target'] = le.fit_transform(df['Topic'].astype(str))
label_map = {i: t for i, t in enumerate(le.classes_)}
print("Label mapping:", label_map)


## 4. Exploratory Data Analysis (EDA)

### 4.1 Year-wise Distribution of BRFSS Records (Fig 2)

In [ ]:
year_col = 'YearStart' if 'YearStart' in df.columns else 'Year'
year_counts = df[year_col].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(year_counts.index.astype(str), year_counts.values,
              color=sns.color_palette('Blues_d', len(year_counts)),
              edgecolor='white', linewidth=0.8)
ax.set_title('Distribution of BRFSS Records by Year', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Number of Records', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('fig2_year_distribution.png', bbox_inches='tight')
plt.show()
print("Fig 2 saved.")


### 4.2 2021 Record Segmentation by Health Domains (Fig 3)

In [ ]:
df_2021 = df[df[year_col] == 2021]
topic_counts = df_2021['Topic'].value_counts()

colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    topic_counts.values, labels=topic_counts.index,
    autopct='%1.1f%%', startangle=140,
    colors=colors, pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold')
ax.set_title('2021 Record Segmentation by Health Domains', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_2021_health_domains.png', bbox_inches='tight')
plt.show()
print("Fig 3 saved.")


### 4.3 Geographic Distribution – Top 10 Locations (Fig 4)

In [ ]:
loc_col = 'LocationDesc' if 'LocationDesc' in df.columns else 'LocationAbbr'
top10_locs = df[loc_col].value_counts().head(10)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top10_locs.index[::-1], top10_locs.values[::-1],
               color=sns.color_palette('Set2', 10))
ax.set_title('Distribution of Classes per Location (Top 10)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Records', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('fig4_top10_locations.png', bbox_inches='tight')
plt.show()
print("Fig 4 saved.")


### 4.4 Year-wise Trends of Cognitive Decline (Fig 5)

In [ ]:
cog_df = df[df['Topic'] == 'Cognitive Decline']
cog_yearly = cog_df[year_col].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cog_yearly.index, cog_yearly.values, marker='o',
        linewidth=2.5, color='#C44E52', markersize=7)
ax.fill_between(cog_yearly.index, cog_yearly.values,
                alpha=0.15, color='#C44E52')
for x, y in zip(cog_yearly.index, cog_yearly.values):
    ax.annotate(f'{y:,}', (x, y), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8)
ax.set_title('Year-wise Trends of Cognitive Decline (2015–2022)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Number of Records')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('fig5_cognitive_decline_trend.png', bbox_inches='tight')
plt.show()
print("Fig 5 saved.")


### 4.5 Top 15 Locations Reporting Cognitive Decline in 2019 (Fig 6)

In [ ]:
cog_2019 = df[(df['Topic'] == 'Cognitive Decline') & (df[year_col] == 2019)]
top15_cog = cog_2019[loc_col].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15_cog.index[::-1], top15_cog.values[::-1],
        color=sns.color_palette('RdYlBu_r', 15))
ax.set_title('Top 15 Locations Reporting Cognitive Decline (2019)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Records')
plt.tight_layout()
plt.savefig('fig6_top15_cognitive_decline_2019.png', bbox_inches='tight')
plt.show()
print("Fig 6 saved.")


### 4.6 Top 10 Questions Related to Cognitive Decline in 2019 (Fig 7)

In [ ]:
q_col = 'Question'
if q_col in df.columns:
    q_avg = (cog_2019.groupby(q_col)['Data_Value']
             .mean().nlargest(10).sort_values())
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(q_avg.index, q_avg.values,
            color=sns.color_palette('viridis', 10))
    ax.set_title('Top 10 Questions Related to Cognitive Decline (2019)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Average Data Value')
    plt.tight_layout()
    plt.savefig('fig7_top10_questions_2019.png', bbox_inches='tight')
    plt.show()
    print("Fig 7 saved.")


### 4.7 Stratification Category Distribution 2019 (Fig 8)

In [ ]:
strat_col = 'StratificationCategory1'
if strat_col in df.columns:
    strat_2019 = df[df[year_col] == 2019][strat_col].value_counts()
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(strat_2019.values, labels=strat_2019.index,
           autopct='%1.1f%%', startangle=90,
           colors=['#4C72B0','#DD8452'],
           wedgeprops=dict(edgecolor='white', linewidth=2))
    ax.set_title('Stratification Category Distribution (2019)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig8_stratification_2019.png', bbox_inches='tight')
    plt.show()
    print("Fig 8 saved.")


### 4.8 Factors Influencing Cognitive Decline – Feature Importance (Fig 9)

In [ ]:
# Quick RF to get feature importances for EDA
feature_cols = [c for c in df.columns if c.endswith('_enc')]
feature_cols += [c for c in ['Data_Value','Comorbidities','CaregivingHours'] if c in df.columns]

X_fi = df[feature_cols].fillna(0)
y_fi = df['Target']

rf_fi = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_fi.fit(X_fi, y_fi)

importance_df = pd.DataFrame({
    'Feature'   : feature_cols,
    'Importance': rf_fi.feature_importances_
}).sort_values('Importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['Feature'], importance_df['Importance'],
        color=sns.color_palette('coolwarm', len(importance_df)))
ax.set_title('Factors Influencing Cognitive Decline (Feature Importance)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('fig9_feature_importance.png', bbox_inches='tight')
plt.show()
print("Fig 9 saved.")


## 5. Classification Models

In [ ]:
def plot_confusion_matrix(cm, class_names, title, filename):
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight')
    plt.show()

CLASS_NAMES = ['Caregiving', 'Cognitive Decline', 'Mental Health', 'Overall Health']
print("Helper function defined.")


### 5.1 Model A – Correlation-Based (Automated) Feature Selection (Fig 10)

In [ ]:
# ── Correlation-based feature selection ──────────────────────────────────────
corr_threshold = 0.1
corr_matrix = df[feature_cols + ['Target']].corr()
corr_with_target = corr_matrix['Target'].drop('Target').abs()
auto_features = corr_with_target[corr_with_target > corr_threshold].index.tolist()
print(f"Features selected by correlation (>{corr_threshold}): {len(auto_features)}")
print(auto_features)

# ── plot correlation with target ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
corr_with_target.sort_values().plot(kind='barh', ax=ax,
                                    color=['#C44E52' if v > corr_threshold
                                           else '#4C72B0'
                                           for v in corr_with_target.sort_values()])
ax.axvline(corr_threshold, color='black', linestyle='--', linewidth=1.2,
           label=f'Threshold = {corr_threshold}')
ax.set_title('Feature Correlation with Target (Health Domain)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('|Pearson Correlation|')
ax.legend()
plt.tight_layout()
plt.savefig('fig10_correlation_feature_selection.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────
X_auto = df[auto_features].fillna(0)
y      = df['Target']

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_auto, y, test_size=0.2, random_state=42, stratify=y)

# ── Random Forest (class_weight=balanced) ─────────────────────────────────────
rf_auto = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                  random_state=42, n_jobs=-1)
rf_auto.fit(X_train_a, y_train_a)
y_pred_a = rf_auto.predict(X_test_a)

print("=== Model A – Correlation-Based Feature Selection ===")
print(f"Overall Accuracy: {accuracy_score(y_test_a, y_pred_a):.2f}")
print("\nClassification Report:")
print(classification_report(y_test_a, y_pred_a, target_names=CLASS_NAMES))


In [ ]:
cm_auto = confusion_matrix(y_test_a, y_pred_a)
plot_confusion_matrix(cm_auto, CLASS_NAMES,
                      'Confusion Matrix – Correlation-Based Model',
                      'fig10b_confusion_auto.png')

# Pretty confusion matrix as DataFrame
cm_df_auto = pd.DataFrame(cm_auto, index=CLASS_NAMES, columns=CLASS_NAMES)
print("\nConfusion Matrix:")
print(cm_df_auto)


### 5.2 Model B – Manual (Domain-Informed) Feature Selection (Fig 11)

In [ ]:
# ── Manually selected features (domain expertise + EDA) ───────────────────────
manual_features = []
candidate_manual = {
    'Question_enc'              : 'Survey question type',
    'Topic_enc_proxy'           : 'Proxy for topic',   # skipped – it IS the target
    'LocationDesc_enc'          : 'Geographic region',
    'YearStart'                 : 'Year',
    'Data_Value'                : 'Observed data value',
    'StratificationCategory1_enc': 'Stratification type (Sex/Race)',
    'Gender_enc'                : 'Gender',
    'Race_enc'                  : 'Race/Ethnicity',
    'Income_enc'                : 'Income level',
    'AgeGroup_enc'              : 'Age group',
    'Comorbidities'             : 'Number of comorbidities',
    'CaregivingHours'           : 'Weekly caregiving hours',
    'Depression_enc'            : 'Depression diagnosis',
    'Hypertension_enc'          : 'Hypertension diagnosis',
    'Diabetes_enc'              : 'Diabetes diagnosis',
    'PhysicalActivity_enc'      : 'Physical activity',
}

manual_features = [f for f in candidate_manual.keys()
                   if f in df.columns and f != 'Topic_enc_proxy']
print(f"Manually selected features ({len(manual_features)}):")
for f in manual_features:
    print(f"  • {f}  →  {candidate_manual.get(f, '')}")


In [ ]:
X_man = df[manual_features].fillna(0)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_man, y, test_size=0.2, random_state=42, stratify=y)

rf_man = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
rf_man.fit(X_train_m, y_train_m)
y_pred_m = rf_man.predict(X_test_m)

print("=== Model B – Manual Feature Selection ===")
print(f"Overall Accuracy: {accuracy_score(y_test_m, y_pred_m):.2f}")
print("\nClassification Report:")
print(classification_report(y_test_m, y_pred_m, target_names=CLASS_NAMES))


In [ ]:
# ── Feature importance plot for manual model (Fig 11) ─────────────────────────
fi_man = pd.DataFrame({
    'Feature'   : manual_features,
    'Importance': rf_man.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_man['Feature'], fi_man['Importance'],
        color=sns.color_palette('viridis', len(fi_man)))
ax.set_title('Manual Feature Selection – Random Forest Feature Importances',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('fig11_manual_feature_importance.png', bbox_inches='tight')
plt.show()


In [ ]:
cm_man = confusion_matrix(y_test_m, y_pred_m)
plot_confusion_matrix(cm_man, CLASS_NAMES,
                      'Confusion Matrix – Manual Feature Selection',
                      'fig11b_confusion_manual.png')

cm_df_man = pd.DataFrame(cm_man, index=CLASS_NAMES, columns=CLASS_NAMES)
print("\nConfusion Matrix:")
print(cm_df_man)


### 5.3 Model C – Random Forest with SMOTE on Manually Selected Features (Fig 12)

In [ ]:
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train_m, y_train_m)

print("Class distribution BEFORE SMOTE:", dict(zip(*np.unique(y_train_m, return_counts=True))))
print("Class distribution AFTER  SMOTE:", dict(zip(*np.unique(y_smote,   return_counts=True))))

rf_smote = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                   random_state=42, n_jobs=-1)
rf_smote.fit(X_smote, y_smote)
y_pred_s = rf_smote.predict(X_test_m)

print("\n=== Model C – Random Forest + SMOTE ===")
print(f"Overall Accuracy: {accuracy_score(y_test_m, y_pred_s):.2f}")
print("\nClassification Report:")
print(classification_report(y_test_m, y_pred_s, target_names=CLASS_NAMES))


In [ ]:
# ── Class distribution bar chart (Fig 12) ─────────────────────────────────────
before_counts = pd.Series(y_train_m).value_counts().sort_index()
after_counts  = pd.Series(y_smote).value_counts().sort_index()
x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, before_counts.values, width, label='Before SMOTE',
       color='#4C72B0', edgecolor='white')
ax.bar(x + width/2, after_counts.values,  width, label='After SMOTE',
       color='#DD8452', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=15, ha='right')
ax.set_title('Class Distribution Before vs After SMOTE',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.legend()
plt.tight_layout()
plt.savefig('fig12_smote_distribution.png', bbox_inches='tight')
plt.show()


In [ ]:
cm_smote = confusion_matrix(y_test_m, y_pred_s)
plot_confusion_matrix(cm_smote, CLASS_NAMES,
                      'Confusion Matrix – Random Forest + SMOTE',
                      'fig12b_confusion_smote.png')

cm_df_smote = pd.DataFrame(cm_smote, index=CLASS_NAMES, columns=CLASS_NAMES)
print("\nConfusion Matrix:")
print(cm_df_smote)


## 6. Model Comparison – Automated vs Manual Feature Selection

In [ ]:
from sklearn.metrics import f1_score

models = {
    'Auto (Correlation)'    : (y_test_a, y_pred_a),
    'Manual (Domain)'       : (y_test_m, y_pred_m),
    'Manual + SMOTE'        : (y_test_m, y_pred_s),
}

comparison = []
for name, (yt, yp) in models.items():
    comparison.append({
        'Model'            : name,
        'Accuracy'         : round(accuracy_score(yt, yp), 4),
        'Macro F1'         : round(f1_score(yt, yp, average='macro'), 4),
        'Weighted F1'      : round(f1_score(yt, yp, average='weighted'), 4),
    })

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

# ── Bar chart comparison ───────────────────────────────────────────────────────
metrics = ['Accuracy', 'Macro F1', 'Weighted F1']
x = np.arange(len(metrics))
width = 0.25
colors = ['#4C72B0', '#55A868', '#DD8452']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (_, row) in enumerate(comp_df.iterrows()):
    ax.bar(x + i*width, [row['Accuracy'], row['Macro F1'], row['Weighted F1']],
           width, label=row['Model'], color=colors[i], edgecolor='white')

ax.set_xticks(x + width); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison – Performance Metrics',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()


## 7. Regional Heatmaps – Cognitive Decline by Year and Location

In [ ]:
cog_pivot = (df[df['Topic'] == 'Cognitive Decline']
             .groupby([year_col, loc_col])['Data_Value']
             .mean()
             .unstack(fill_value=0))

# keep top 20 locations for readability
top20 = cog_pivot.sum().nlargest(20).index
cog_pivot_top20 = cog_pivot[top20]

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(cog_pivot_top20.T, cmap='YlOrRd', linewidths=0.3,
            linecolor='white', ax=ax, annot=False,
            cbar_kws={'label': 'Avg. Data Value'})
ax.set_title('Heatmap – Cognitive Decline by Year & Location (Top 20)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Location')
plt.tight_layout()
plt.savefig('heatmap_cognitive_decline.png', bbox_inches='tight')
plt.show()


## 8. Conclusion

### Key Findings

| Finding | Detail |
|---------|--------|
| **Cognitive Decline Peak** | Sharp spike in 2019 (≈ 4,773 records) – linked to improved reporting & awareness |
| **Regional Disparities** | South, Midwest, Northeast, West show consistently higher rates |
| **Demographic Gaps** | Women and minorities disproportionately affected |
| **Survey Design Matters** | `Question` and `Topic` features outweigh demographic variables in classification |
| **Best Model** | Manual feature selection RF → **92% accuracy**, F1 = 1.00 (Cognitive Decline) |
| **SMOTE Effect** | Improved minority-class recall; slight trade-off in overall accuracy |

### Recommendations
1. Integrate culturally sensitive, region-specific screening into primary care
2. Prioritise caregiving support in rural / low-income communities  
3. Standardise and improve survey instrument design for public-health data collection  
4. Use data-driven models for proactive, equitable resource allocation  
5. Track post-pandemic cognitive trends longitudinally

---
*Notebook recreated from published research paper – all figures correspond to the paper's Fig 2–12.*
